In [7]:
from sys import platform
from pathlib import Path
import os
from requests_html import HTMLSession
import wget
import imageio.v3 as iio

#### System Paths

In [8]:
if platform == 'linux':
    home_dir = os.path.expanduser('~')
elif platform == 'win32':
    home_dir = os.path.expandvars(r'%HOMEDRIVE%%HOMEPATH%\OneDrive') #Stupid OneDrive
else:
    raise ValueError("Cannot set home directory for this OS.")
print(home_dir)

/home/monorhesus


#### Output directories

In [12]:
img_dir = os.path.join('Pictures', 'visible-human', 'raw')
img_dir

'Pictures/visible-human/raw'

In [13]:
wk_dir = os.path.join(home_dir, img_dir)
wd_path = Path(wk_dir)
wd_path.mkdir(parents=True, exist_ok=True)
print(f'Working dir: {wk_dir}')

Working dir: /home/monorhesus/Pictures/visible-human/raw


#### Get top-level index URLs

In [7]:
index_url = 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/index.html'

In [8]:
session = HTMLSession()

In [9]:
i = session.get(index_url)
urls_all = i.html.absolute_links
index_urls = [u for u in urls_all if 'radiological' not in u]
index_urls

['https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/head/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/abdomen/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/pelvis/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/thorax/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/thighs/index.html',
 'https://data.lhncbc.nlm.nih.gov/public/Visible-Human/Male-Images/PNG_format/legs/index.html']

#### Get images

In [36]:
for url in index_urls:
    r = session.get(url)
    img_urls = list(r.html.absolute_links)
    img_urls[:5]
    
    # Create body part directory
    body_part = list(set([u.split('/')[-2] for u in img_urls]))
    assert len(body_part) == 1, 'Error parsing body part'
    body_dirname = body_part[0]
    body_dir = os.path.join(wk_dir, body_dirname)
    mkbody_dir = Path(body_dir)
    mkbody_dir.mkdir(parents=True, exist_ok=True)
    print(f'{body_part[0].upper()}')
    
    for img_url in tqdm(img_urls, desc='Download Progress', total=len(img_urls)): 
        if img_url.endswith('.png'):
            
            # Avoid wget duplicates
            img_name = img_url.split('/')[-1]
            filename = rf'{body_dir}\{img_name}'
            if os.path.exists(filename):
                # print(f'{filename} exists, skipping')
                pass
            else:
                # Download image
                wget.download(img_url, out=body_dir)
                # print(f'\nDownloaded {body_dirname}/{img_name}')

HEAD


Download Progress: 100%|██████████████████████████████████████████████████████████| 377/377 [00:00<00:00, 15095.20it/s]


ABDOMEN


Download Progress: 100%|██████████████████████████████████████████████████████████| 543/543 [00:00<00:00, 16367.05it/s]


PELVIS


...

Download Progress: 100%|██████████████████████████████████████████████████████████| 409/409 [00:00<00:00, 16987.71it/s]


THIGHS


Download Progress: 100%|██████████████████████████████████████████████████████████| 680/680 [00:00<00:00, 17190.08it/s]


LEGS


Download Progress: 100%|██████████████████████████████████████████████████████████| 614/614 [00:00<00:00, 15790.49it/s]


#### Animate segmentation
Takes a while and generates a 2.6Gb gif, see other script for other methods.

In [58]:
sorted_imgs = {}
for root, dirs, files in os.walk(wk_dir):
    for file in files:
        k = int(''.join([c for c in file if c.isdigit()]))
        v = os.path.join(root, file)
        sorted_imgs[k] = v
sorted_imgs = {k: sorted_imgs[k] for k in sorted(sorted_imgs)} # Sort images 
[sorted_imgs[k] for k in list(sorted_imgs.keys())[:10]] # Verify proper order for axial segmentation

['C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1001.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1002.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1003.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1004.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1005.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1006.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1007.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1008.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1009.png',
 'C:\\Users\\th1si\\OneDrive\\Pictures\\visible-human\\head\\a_vm1010.png']

In [66]:
filenames = [sorted_imgs[k] for k in sorted_imgs] 
images = [iio.imread(f) for f in filenames]
iio.imwrite(f'{wk_dir}\\gif-visiblehuman-axial.gif', images, duration=25, loop=0)